# Federated privacy trade-off: utility, privacy and communication on UCI Adult (notebook-scale)

This notebook loads UCI Adult / Census Income (the 1994 U.S. Census extraction, 48,842 rows),
builds the three workclass clients and the IID control, runs the centralised SVM, Random
Forest and Decision Tree baselines, local-only training and sample-weighted FedAvg through the
repository's Flower apps, then compares IID with non-IID clients, sweeps the privacy budget
ε over {1, 3, 8, no noise}, shows per-client performance and communication cost, and ends with
the benchmark's Before / After table, grouped bar chart and ε-versus-accuracy chart.

The files are downloaded by `make data` from the canonical UCI source
(http://archive.ics.uci.edu/ml/datasets/Adult) and are never committed.

The method follows the federated learning approach with differential privacy described by
Zhang et al. (2025).

## What this notebook does and does not show

**It shows**

- the per-client sizes and label rates of the workclass split and the IID control;
- a **notebook-scale run** of the main benchmark suite: the full protocol (20 rounds, all
  methods, both client splits, the ε sweep) on fewer seeds than the reference run. Every
  mean ± standard deviation is over the seeds set in the parameters cell; with fewer than five
  seeds a spread is only indicative;
- what federation buys the clients (FedAvg versus local-only), what not pooling the data costs
  (FedAvg versus the centralised MLP of the same architecture) and what privacy costs (DP
  versus no DP).

**It does not show**

- the reference numbers. Those come from the five-seed reference run and are in
  `experiments/results/RESULTS.md`; this notebook writes nothing under
  `experiments/results/reference/`;
- anything about real organisations: the three clients are simulated domains derived from one
  public census dataset, not real employers, hospitals or banks. The client construction, the
  IID control, the 20-round protocol and the ε sweep are `[Assumption — not in paper]`;
- a fairness certification: the per-client gap is descriptive, and the two smaller domains
  have only about two thousand test rows each, so it is noisy;
- a deployment-grade privacy implementation (see the ε section for exactly what ε covers).

## Parameters

Every knob of this notebook is in the next cell.

In [ ]:
import os
from pathlib import Path

FEDERATED_CONFIG = "configs/federated_adult.yaml"
BEFORE_AFTER_CONFIG = "configs/before_after.yaml"

# Dotted overrides on top of FEDERATED_CONFIG, e.g. ["training.rounds=10"] for a faster,
# NON-protocol run (DP noise is then calibrated for 10 rounds).
OVERRIDES = []

# Scale choice: 2 of the 5 reporting seeds with the full protocol otherwise (20 rounds, all six
# centralised baselines, local-only, FedAvg, the three DP conditions, both client splits). The
# RBF SVM is fitted on all 32,561 training rows, exactly as configured (no subsample): it is
# the slowest single task but runs in parallel with the federated tasks, so the notebook stays
# within a few minutes. DP-SGD runs (per-sample gradients) dominate the rest of the run time.
# Fewer seeds is the only reduction, so no method is helped or handicapped by the scale.
SEEDS = [0, 1]

# Worker processes for the benchmark tasks. Results do not depend on this number.
WORKERS = max(1, min(4, os.cpu_count() or 1))

# Seed used for the partition statistics table (the workclass split does not depend on it).
STATS_SEED = SEEDS[0]

OUTPUT_DIR = Path("experiments/results/runs/notebooks/01_federated_privacy_tradeoff")

In [ ]:
import os
from pathlib import Path

# The runner (python -m experiments.run_notebooks) starts the kernel in the repository root.
# If you opened the notebook from the notebooks/ folder instead, move up one level.
if not Path("configs").is_dir() and Path("../configs").is_dir():
    os.chdir("..")
assert Path("configs").is_dir(), "Run this notebook with the repository root as working directory."

from IPython.display import Image, Markdown, display


def provenance(config_paths, overrides, seeds, extra=None):
    """Print where every number below comes from: config file(s), overrides and seeds."""
    paths = [config_paths] if isinstance(config_paths, (str, Path)) else list(config_paths)
    print("config:    " + ", ".join(str(path) for path in paths))
    print("overrides: " + (", ".join(overrides) if overrides else "none"))
    print(f"seeds:     {list(seeds)} ({len(seeds)} seed(s); notebook-scale run, not the reference run)")
    if extra:
        print(extra)


def show_table(header, body, caption=""):
    """Render a small Markdown table (cells are already formatted strings)."""
    lines = ["| " + " | ".join(header) + " |", "|" + "|".join(["---"] * len(header)) + "|"]
    lines += ["| " + " | ".join(str(cell) for cell in row) + " |" for row in body]
    display(Markdown((caption + "\n\n" if caption else "") + "\n".join(lines)))

In [ ]:
%matplotlib inline
import math
import multiprocessing
import time
from collections import defaultdict

import matplotlib.pyplot as plt

from experiments.build_results import MetricCfg, find_row, measurement
from experiments.run_federated_benchmark import SUMMARY_KEYS, dp_tradeoff_rows
from src.common import plotting
from src.common.before_after import format_value, make_row, render_markdown, write_rows_csv
from src.common.config import load_config
from src.common.io import write_csv
from src.common.labels import REPRODUCED
from src.common.metrics import format_mean_std, summarise_by_seed
from src.common.plotting import Bar, BeforeAfterPanel, bar_panel, before_after_figure
from src.federated import GUARANTEE, load_dataset, load_federated_config, make_partition, partition_statistics
from src.federated.partition import PARTITION_STATS_FIELDS
from src.federated.runner import (
    FINAL_FIELDS, FINAL_METRICS, PER_CLIENT_FIELDS, PER_ROUND_FIELDS, build_tasks, run_task,
)

plotting.apply_style()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
config, raw = load_federated_config(FEDERATED_CONFIG, overrides=OVERRIDES)
TRACK = load_config(BEFORE_AFTER_CONFIG)["federated"]
METRICS = [MetricCfg(**entry) for entry in TRACK["metrics"]]
BY_KEY = {metric.key: metric for metric in METRICS}
LABELS = TRACK["method_labels"]
SPLITS = TRACK["splits"]
PRIMARY = next(iter(SPLITS))
INDEPENDENT = set(TRACK["split_independent_methods"])
DATASET_LINE = TRACK["dataset_line"]
RUN_LABEL = f"notebook-scale run, {len(SEEDS)} seeds, {config.training.rounds} rounds"

# Fail early, with instructions, if the Adult files are not there.
try:
    dataset = load_dataset(config.dataset, config.evaluation, STATS_SEED)
except FileNotFoundError as error:
    raise FileNotFoundError(
        f"UCI Adult is not available: {error}\n"
        "Run `make data` from the repository root (it downloads the files from the canonical UCI "
        "source into data/adult/ and verifies their checksums), then run this notebook again."
    ) from None

provenance(FEDERATED_CONFIG, OVERRIDES, SEEDS)
print(DATASET_LINE)
print(f"training rows {len(dataset.train)}, evaluation rows ({dataset.eval_on}) {len(dataset.test)}, "
      f"features {dataset.num_features}, clients {dataset.domain_names}")
print(f"rounds {config.training.rounds}, local epochs {config.training.local_epochs}, "
      f"aggregation {config.federation.aggregation_rule}, DP sweep {[e.name for e in config.dp.sweep]}, delta {config.dp.delta}")

## Clients: sizes and label rates

Dataset: UCI Adult / Census Income (1994 U.S. Census extraction; 48,842 rows; workclass clients Private / Government / Self-employed plus IID control)

The client construction is `[Assumption — not in paper]`: it is chosen by this repository. The
workclass split puts the training rows of workclass Private into one client,
Federal-gov, State-gov and Local-gov into "Government", and Self-emp-inc and Self-emp-not-inc
into "Self-employed"; rows with workclass `?`, Without-pay or Never-worked belong to no client
(the `(excluded)` row) and are used only by the centralised baselines trained on all rows. The
IID control deals the same client rows, shuffled, into three equal clients.

How to read it: the workclass clients differ in size and in the share of the positive class
(income >50K) — that is the non-IID structure. The IID clients have equal sizes and nearly
equal label rates by construction.

In [ ]:
provenance(FEDERATED_CONFIG, OVERRIDES, [STATS_SEED], extra="partition statistics (training rows only)")
stats = []
for split in config.partitions.splits:
    natural = split == dataset.natural_split_name
    partition = make_partition(split, dataset, config.partitions, STATS_SEED, None)
    stats.extend(partition_statistics(partition, dataset, STATS_SEED))
write_csv(OUTPUT_DIR / "partition_stats.csv", stats, PARTITION_STATS_FIELDS)
show_table(
    ["Split", "Client", "Rows", "Positives (>50K)", "Positive rate", "Share of client rows", "Composition"],
    [[s["split"], s["client"], s["rows"], s["positives"], f"{s['positive_rate']:.4f}",
      f"{s['share_of_client_rows']:.4f}" if s["share_of_client_rows"] != "" else "", s["composition"]] for s in stats],
    caption="**Per-client training rows and label rates** (computed from the files on disk).",
)
clients = [s for s in stats if s["client"] != "(excluded)"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
labels = [f"{s['split']}: {s['client']}" for s in clients]
colors = [plotting.SPLIT_COLORS.get(s["split"], plotting.REFERENCE_COLOR) for s in clients]
bar_panel(axes[0], labels, [s["rows"] for s in clients], [None] * len(clients), colors, "training rows", "Rows per client", digits=0)
bar_panel(axes[1], labels, [s["positive_rate"] for s in clients], [None] * len(clients), colors,
          "fraction of rows with income >50K", "Positive-class rate per client", digits=3)
fig.suptitle(f"UCI Adult client sizes and label rates {REPRODUCED} — notebook-scale, seed {STATS_SEED}",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.text(0.01, 0.002, "Violet = workclass clients (non-IID), aqua = IID control. Simulated clients derived from one "
         "public 1994 census dataset; training rows only.", fontsize=8.5, color=plotting.INK_SECONDARY)
fig.tight_layout(rect=(0, 0.04, 1, 0.93))
fig.savefig(OUTPUT_DIR / "client_sizes_label_rates.png", dpi=120)
plt.show()

## Run the main suite

Dataset: UCI Adult / Census Income (1994 U.S. Census extraction; 48,842 rows; workclass clients Private / Government / Self-employed plus IID control)

The client construction is `[Assumption — not in paper]`. The next cell builds the tasks of the
benchmark's `main` suite (`build_tasks`) and runs each with `run_task` — the same functions
`experiments/run_federated_benchmark.py` uses — in `WORKERS` processes: per seed, the
centralised baselines once, and for each client split local-only training, weighted FedAvg and
weighted FedAvg with DP-SGD at each target ε. Federated conditions go through the real Flower
`ServerApp` / `ClientApp` on the in-process grid. Every metric is computed on the shared Adult
test set (16,281 rows), and the final round is reported, never the best round.

In [ ]:
provenance(FEDERATED_CONFIG, OVERRIDES, SEEDS, extra=f"workers: {WORKERS}")
tasks = build_tasks(config, "main", SEEDS)
print(f"{len(tasks)} tasks: " + ", ".join(sorted({f'{t.kind}:{t.condition}' for t in tasks})))
started = time.perf_counter()
if WORKERS > 1:
    with multiprocessing.get_context("spawn").Pool(processes=min(WORKERS, len(tasks))) as pool:
        outputs = pool.starmap(run_task, [(raw, task) for task in tasks], chunksize=1)
else:
    outputs = [run_task(raw, task) for task in tasks]
print(f"finished in {time.perf_counter() - started:.0f} s")

per_round, per_client, final = [], [], []
for output in outputs:
    per_round.extend(output.per_round)
    per_client.extend(output.per_client)
    final.extend(output.final)
for row in final:
    row.setdefault("shared_across_splits", "")
summary = summarise_by_seed(final, SUMMARY_KEYS, FINAL_METRICS)
write_csv(OUTPUT_DIR / "per_round.csv", per_round, PER_ROUND_FIELDS)
write_csv(OUTPUT_DIR / "per_client.csv", per_client, PER_CLIENT_FIELDS)
write_csv(OUTPUT_DIR / "final.csv", final, FINAL_FIELDS)


def row_of(split, method):
    found = find_row(summary, split=split, method=method)
    if found is None and method in INDEPENDENT:
        found = find_row(summary, method=method)
    return found


def value(split, method, key):
    return measurement(row_of(split, method), BY_KEY[key])


def cell(split, method, key):
    return format_value(value(split, method, key), BY_KEY[key].digits)

## Centralised baselines, local-only training and weighted FedAvg

Dataset: UCI Adult / Census Income (1994 U.S. Census extraction; 48,842 rows; workclass clients Private / Government / Self-employed plus IID control)

The client construction is `[Assumption — not in paper]`. All rows below use the workclass
clients where a split matters. The centralised baselines are trained once per seed on all
32,561 training rows (including the rows that belong to no client);
`centralised_mlp_client_rows` is the same MLP trained on the client rows only. Local-only is
the mean over the three clients' own models, each trained alone for the same number of passes.

How to read it: the federated model is a small MLP, and tree ensembles are strong on Adult, so
compare FedAvg with the centralised **MLP** (the cost of not pooling the data) and with
**local-only** (whether collaborating helps the clients), not only with the Random Forest. The
majority-class row is the accuracy of predicting income <=50K for everyone: any useful model
must be clearly above it.

In [ ]:
provenance(FEDERATED_CONFIG, OVERRIDES, SEEDS, extra=f"clients: {SPLITS[PRIMARY]}")
methods = ["svm", "random_forest", "decision_tree", "centralised_mlp", "centralised_mlp_client_rows",
           "majority_class", "local_only", "fedavg"]
keys = ["accuracy", "f1", "precision", "recall", "fairness_gap_accuracy"]
show_table(
    ["Method", *[f"{BY_KEY[k].label} ({BY_KEY[k].unit})" for k in keys]],
    [[LABELS.get(m, m), *[cell(PRIMARY, m, k) for k in keys]] for m in methods],
    caption=f"**{DATASET_LINE}**\n\n**Baselines, local-only and FedAvg — {SPLITS[PRIMARY]}, {RUN_LABEL} {REPRODUCED}.** "
            "Mean ± sample std across seeds on the shared test set. Reference numbers: `experiments/results/RESULTS.md`.",
)

## IID versus non-IID clients

Dataset: UCI Adult / Census Income (1994 U.S. Census extraction; 48,842 rows; workclass clients Private / Government / Self-employed plus IID control)

The client construction is `[Assumption — not in paper]`. The same training rows feed both
splits; only the way they are dealt to the clients differs.

How to read it: with IID clients every client sees the same distribution, so FedAvg is
expected to do at least as well as with the natural workclass clients; a large gap between the
two columns is the price of heterogeneity. Local-only usually suffers more from the split,
because the small Self-employed and Government clients train on few rows.

In [ ]:
provenance(FEDERATED_CONFIG, OVERRIDES, SEEDS)
cond = ["local_only", "fedavg", *[entry.name for entry in config.dp.sweep]]
show_table(
    ["Method", *[f"Accuracy, {label}" for label in SPLITS.values()], *[f"F1, {label}" for label in SPLITS.values()],
     *[f"Fairness gap, {label}" for label in SPLITS.values()]],
    [[LABELS.get(m, m), *[cell(s, m, "accuracy") for s in SPLITS], *[cell(s, m, "f1") for s in SPLITS],
      *[cell(s, m, "fairness_gap_accuracy") for s in SPLITS]] for m in cond],
    caption=f"**{DATASET_LINE}**\n\n**IID versus non-IID — {RUN_LABEL} {REPRODUCED}.** Mean ± sample std across seeds.",
)

## Privacy: the ε sweep {1, 3, 8, no noise}

Dataset: UCI Adult / Census Income (1994 U.S. Census extraction; 48,842 rows; workclass clients Private / Government / Self-employed plus IID control)

The client construction and the ε sweep are `[Assumption — not in paper]`. Each DP condition
runs DP-SGD (per-sample clipping, Gaussian noise, Poisson sampling) inside every client through
Opacus; each client's noise multiplier is calibrated from its own sampling rate and total step
count so that it spends the target ε over all rounds, accounted with the RDP accountant; the
reported ε is the maximum over clients.

What ε covers (printed in full by the next cell from `GUARANTEE` in
`src/federated/privacy.py`): record-level (ε, δ)-DP of each client's training rows with respect
to the sequence of model updates that client sends, against an honest-but-curious server, for
that run alone. It does **not** cover client-level privacy, the row counts each client reports
in the clear or the quantities derived from them, hyperparameter tuning, or the joint release of several runs on the same rows —
this notebook publishes several runs on the same public rows, which would not be acceptable for
private data. The DP randomness of each client comes from a client-held secret that never
appears in a message; the benchmark harness fixes that secret for reproducibility, which a
deployment must not do. Opacus `secure_mode` is off: this is a research simulation.

How to read it: utility is expected to fall as ε shrinks, but at the larger budgets the drop
can be smaller than the seed-to-seed spread, so the curve need not be monotonic on a few seeds
(a DP run can even score slightly above the run without noise on some metric; that is noise,
not a benefit of privacy). Compare the size of each drop with the spread, and the DP runs with the centralised MLP and the
majority-class line in the chart. The y axes of the ε chart do not start at zero (it is a line
chart, not a bar chart): read the values, not the slopes.

In [ ]:
provenance(FEDERATED_CONFIG, OVERRIDES, SEEDS, extra=f"delta = {config.dp.delta}")
print("Guarantee:", GUARANTEE)
tradeoff = dp_tradeoff_rows(config, final, per_client)
show_table(
    ["Clients", "Condition", "Target ε", "Achieved ε (mean of per-run max)", "Achieved ε (max)",
     "Noise multiplier per client", "Accuracy", "F1 (class >50K)", "Seeds"],
    [[SPLITS.get(r["split"], r["split"]), r["condition"], r["target_epsilon"],
      f"{r['achieved_epsilon_mean']:.3f}", f"{r['achieved_epsilon_max']:.3f}", r["noise_multiplier_per_client"] or "—",
      format_mean_std(r["accuracy_mean"], r["accuracy_std"], 4, r["n_seeds"]),
      format_mean_std(r["f1_mean"], r["f1_std"], 4, r["n_seeds"]), r["n_seeds"]] for r in tradeoff],
    caption=f"**{DATASET_LINE}**\n\n**ε sweep — {RUN_LABEL} {REPRODUCED}.** ∞ / inf = no noise, no privacy guarantee.",
)

In [ ]:
provenance(FEDERATED_CONFIG, OVERRIDES, SEEDS)
conditions = TRACK["epsilon_conditions"]
positions = list(range(len(conditions)))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, key in zip(axes, ("accuracy", "f1")):
    for index, (split, split_label) in enumerate(SPLITS.items()):
        measured = [value(split, c["method"], key) for c in conditions]
        if not all(m.available for m in measured):
            continue
        means = [m.mean for m in measured]
        errors = [0.0 if m.std is None or math.isnan(m.std) else m.std for m in measured]
        ax.errorbar(positions, means, yerr=errors, color=plotting.SPLIT_COLORS.get(split, plotting.REFERENCE_COLOR),
                    marker=plotting.MARKERS[index], capsize=3, label=f"Weighted FedAvg, {split_label}",
                    markeredgecolor=plotting.SURFACE, markeredgewidth=1.2)
        for x, mean in zip(positions, means):
            ax.annotate(f"{mean:.3f}", (x, mean), xytext=(-18 if index == 0 else 18, 7 if index == 0 else -13),
                        textcoords="offset points", ha="center", fontsize=7.5,
                        color=plotting.SPLIT_COLORS.get(split, plotting.INK))
    references = [("centralised_mlp", "Centralised MLP (no privacy)", (0, (5, 3)))]
    if key == "accuracy":
        references.append(("majority_class", "Majority class (always <=50K)", (0, (1, 2))))
    for method, name, dashes in references:
        ref = value(PRIMARY, method, key)
        if ref.available:
            ax.axhline(ref.mean, color=plotting.REFERENCE_COLOR, linestyle=dashes, linewidth=1.3, label=name)
    ax.set_xticks(positions)
    ax.set_xticklabels([c["label"] for c in conditions])
    ax.set_xlim(-0.5, len(conditions) - 0.5)
    ax.set_xlabel("Privacy budget ε per client (smaller = more private; ∞ = no noise)")
    ax.set_ylabel(f"{BY_KEY[key].label} on the Adult test set")
    ax.set_title(BY_KEY[key].label, loc="left")
    ax.legend(fontsize=7.5, loc="lower right")
fig.suptitle(f"ε versus accuracy and F1 on UCI Adult {REPRODUCED} — {RUN_LABEL}", x=0.01, ha="left",
             fontsize=12, fontweight="bold")
fig.text(0.01, 0.002, "UCI Adult / Census Income (1994 U.S. Census extraction); mean ± sample std across seeds; record-level "
         "ε per client. The y axes do not start at zero.", fontsize=8.5, color=plotting.INK_SECONDARY)
fig.tight_layout(rect=(0, 0.04, 1, 0.93))
fig.savefig(OUTPUT_DIR / "epsilon_tradeoff.png", dpi=120)
plt.show()

## Per-client performance

Dataset: UCI Adult / Census Income (1994 U.S. Census extraction; 48,842 rows; workclass clients Private / Government / Self-employed plus IID control)

The client construction is `[Assumption — not in paper]`. Every model is evaluated on the rows
of the shared test set that belong to each workclass domain (Private, Government,
Self-employed), computed identically for every method. For local-only, each domain's value is
that client's own model. The fairness gap in the earlier tables is the largest minus the
smallest of these accuracies.

How to read it: check whether federation helps the small clients (Government and
Self-employed versus local-only), and whether DP costs the small clients more than the large
one. Note that a local-only value is a specialised model scored on its own home domain: it can
match or beat the shared FedAvg model there even when the same local models are clearly worse
on the full test set (the overall tables above). Both readings are true at once; report the
one that answers your question. The two smaller domains have only about two thousand test rows each, so differences of a
fraction of a percentage point are noise.

In [ ]:
provenance(FEDERATED_CONFIG, OVERRIDES, SEEDS, extra=f"clients: {SPLITS[PRIMARY]}; final round")
rounds = config.training.rounds
baseline_methods = set(config.baselines.methods)
selected = [
    r for r in per_client
    if r["split"] == PRIMARY and r["domain"] and r["domain_accuracy"] != ""
    and int(r["round"]) == (0 if r["condition"] in baseline_methods else rounds)
]
per_domain = summarise_by_seed(selected, ["condition", "domain"], ["domain_accuracy", "domain_f1", "domain_n"])
domain_names = list(dataset.domain_names)
chart_methods = TRACK["chart_methods"]


def domain_value(method, domain, key):
    row = find_row(per_domain, condition=method, domain=domain)
    return (row[f"{key}_mean"], row[f"{key}_std"], row["n_seeds"]) if row else (math.nan, math.nan, 0)


show_table(
    ["Method", *[f"Accuracy, {d} (n={int(domain_value('fedavg', d, 'domain_n')[0])})" for d in domain_names],
     *[f"F1, {d}" for d in domain_names]],
    [[LABELS.get(e["method"], e["method"]),
      *[format_mean_std(*domain_value(e["method"], d, "domain_accuracy")[:2], 4, domain_value(e["method"], d, "domain_accuracy")[2]) for d in domain_names],
      *[format_mean_std(*domain_value(e["method"], d, "domain_f1")[:2], 4, domain_value(e["method"], d, "domain_f1")[2]) for d in domain_names]] for e in chart_methods],
    caption=f"**{DATASET_LINE}**\n\n**Per-client (per-domain) test performance — {SPLITS[PRIMARY]}, {RUN_LABEL} {REPRODUCED}.**",
)
fig, axes = plt.subplots(1, len(domain_names), figsize=(5.4 * len(domain_names), 4.2), squeeze=False)
for ax, domain in zip(axes[0], domain_names):
    values = [domain_value(e["method"], domain, "domain_accuracy") for e in chart_methods]
    bar_panel(ax, [e["short"] for e in chart_methods], [v[0] for v in values], [v[1] for v in values],
              [plotting.ROLE_COLORS[e["role"]] for e in chart_methods], "accuracy on this domain's test rows",
              f"{domain}", digits=3)
fig.suptitle(f"Per-client accuracy, {SPLITS[PRIMARY]} {REPRODUCED} — {RUN_LABEL}", x=0.01, ha="left",
             fontsize=12, fontweight="bold")
fig.text(0.01, 0.002, "Blue = Before methods (centralised and local-only), orange = After methods (weighted FedAvg, with and "
         "without DP). Error bars = sample std across seeds. Simulated clients of one public dataset.", fontsize=8.5,
         color=plotting.INK_SECONDARY)
fig.tight_layout(rect=(0, 0.04, 1, 0.93))
fig.savefig(OUTPUT_DIR / "per_client_accuracy.png", dpi=120)
plt.show()

## Communication cost

Dataset: UCI Adult / Census Income (1994 U.S. Census extraction; 48,842 rows; workclass clients Private / Government / Self-employed plus IID control)

The client construction is `[Assumption — not in paper]`. Communication counts training
traffic only: the bytes of the model parameters sent down to and up from every client in every round,
counted from the Flower messages. It is zero for methods that never exchange a model (centralised
baselines, local-only). Client-side evaluation is off for every condition, so there is no
evaluation traffic.

How to read it: the cost depends on the model size, the number of clients and the number of
rounds, not on DP — DP conditions send the same model shape as FedAvg. It is a property of the
protocol, not something the method improves.

In [ ]:
provenance(FEDERATED_CONFIG, OVERRIDES, SEEDS)
bytes_metric = BY_KEY["bytes_total"]
show_table(
    ["Method", *[f"{bytes_metric.label}, {label} ({bytes_metric.unit})" for label in SPLITS.values()],
     "Rounds", "Clients"],
    [[LABELS.get(e["method"], e["method"]), *[cell(s, e["method"], "bytes_total") for s in SPLITS],
      next((r["rounds"] for r in final if r["method"] == e["method"]), ""),
      next((r["num_clients"] for r in final if r["method"] == e["method"]), "")] for e in chart_methods],
    caption=f"**{DATASET_LINE}**\n\n**Communication over all rounds — {RUN_LABEL} {REPRODUCED}.**",
)

## Before / After — the federated benchmark, computed from this notebook's run

Dataset: UCI Adult / Census Income (1994 U.S. Census extraction; 48,842 rows; workclass clients Private / Government / Self-employed plus IID control)

The client construction is `[Assumption — not in paper]`. The pairs, metrics, units,
directions, near-zero thresholds and display names are read from `configs/before_after.yaml`,
exactly as `experiments/build_results.py` reads them for the reference tables; the rows are
built with `make_row` and rendered with `render_markdown` (sign convention printed under the
tables). Weighted FedAvg without DP is the After method relative to the centralised and
local-only baselines, and the Before method relative to the DP runs.

How to read it: a positive Improvement % means better after; **a negative Improvement % is a
regression and is reported as one** — for example, FedAvg may well be worse than the Random
Forest, and every DP run is expected to be worse than FedAvg without DP on utility while better
on the privacy budget. Rows where the Before value is zero (communication of methods that never
communicate) or infinite (ε without DP) show "N/A — baseline zero / not comparable" instead of
a percentage. This is a notebook-scale table with the number of seeds stated; the reference
table is in `experiments/results/RESULTS.md`.

In [ ]:
provenance([FEDERATED_CONFIG, BEFORE_AFTER_CONFIG], OVERRIDES, SEEDS)
rows_by_split = {
    split: [make_row(metric.spec, LABELS[before], value(split, before, metric.key), LABELS[after],
                     value(split, after, metric.key))
            for metric in METRICS for before, after in TRACK["pairs"]]
    for split in SPLITS
}
for split, split_label in SPLITS.items():
    write_rows_csv(rows_by_split[split], OUTPUT_DIR / f"before_after_{split}.csv",
                   track=f"federated:{split} (notebook-scale)")
    display(Markdown(
        f"{DATASET_LINE}\n\n**Before / After — clients: {split_label}; {RUN_LABEL}; shared Adult test set "
        f"(16,281 rows); source label {REPRODUCED} (this notebook's run).** Reference numbers: "
        "`experiments/results/RESULTS.md`.\n\n"
        + render_markdown(rows_by_split[split])
    ))

In [ ]:
provenance([FEDERATED_CONFIG, BEFORE_AFTER_CONFIG], OVERRIDES, SEEDS)
for split, split_label in SPLITS.items():
    panels = []
    for metric in METRICS:
        bars = []
        for entry in chart_methods:
            measured = value(split, entry["method"], metric.key)
            bars.append(Bar(entry["short"], entry["role"], measured.mean if measured.available else None, measured.std))
        panels.append(BeforeAfterPanel(metric.label, metric.unit, metric.spec.direction, tuple(bars), min(metric.digits, 3)))
    chart = before_after_figure(
        panels,
        OUTPUT_DIR / f"before_after_{split}.png",
        f"{TRACK['title']} ({split_label}): Before / After — {RUN_LABEL}",
        REPRODUCED,
        caption=(
            f"UCI Adult / Census Income (1994 U.S. Census extraction), shared test set; {config.training.rounds} rounds; bars: "
            f"mean across {len(SEEDS)} seeds; error bars: sample std (indicative only with fewer than five seeds). "
            "∞ = no privacy guarantee. Communication is zero for methods that never exchange models. "
            "Reference run: experiments/results/RESULTS.md."
        ),
        columns=3,
    )
    display(Image(filename=str(chart)))

## Interpretation and limits

- **Notebook scale.** Every table and chart above is a notebook-scale run over the seeds set in
  the parameters cell (the number is printed with each). The reference numbers, from five
  seeds, are in `experiments/results/RESULTS.md`. With fewer than five seeds a spread is only
  indicative.
- **Read the comparisons that mean something.** FedAvg versus local-only answers whether the
  clients gain from collaborating; FedAvg versus the centralised MLP of the same architecture
  shows what not pooling the data costs; DP versus no DP shows what privacy costs. FedAvg is a
  small MLP: if the Random Forest or Decision Tree is better, the Before / After table shows a
  negative Improvement % and that is the result — federated averaging is not an accuracy
  improvement over strong centralised models unless a row says so.
- **Non-IID.** The gap between the workclass and IID columns is the cost of heterogeneous
  clients in this simulation. The clients are simulated domains derived from one public 1994
  census dataset, not real organisations, and the whole client design is
  `[Assumption — not in paper]`.
- **Privacy.** ε is record-level, per client, for one run, against an honest-but-curious
  server; it does not cover client-level privacy, the reported row counts and their derived quantities, tuning, or the joint
  release of the several runs this notebook makes on the same rows. DP randomness comes from a
  client-held secret that never appears in a message; the harness fixes it for
  reproducibility, which a deployment must not do. Opacus `secure_mode` is off. The utility drop measured here from ε = 8 to ε = 1 on this dataset says nothing about other datasets or models.
- **Per-client numbers and the fairness gap are descriptive** and noisy for the two small
  domains; age, race and sex are never model inputs, but `relationship` and `marital-status`
  are, and they are proxies for sex.